In [1]:
import pandas as pd

df = pd.read_csv("6.geolocalizados_completo_1_ronda.csv")

In [ ]:
# import pandas as pd

# # =============================================
# # PASO 1: Crear el sub_df con id, confidence_score y pdf_links_marcia
# # =============================================
# sub_df = df[['id', 'confidence_score', 'pdf_links_marcia']].copy()

# print(f"Sub_df inicial creado: {len(sub_df):,} filas")

# # =============================================
# # PASO 2: EXPLOTAR los enlaces (una fila por cada PDF)
# #    → Si un id tiene 3 links separados por ';', ahora tendrá 3 filas
# # =============================================

# sub_df['pdf_links_marcia'] = sub_df['pdf_links_marcia'].fillna('').astype(str).str.strip()

# sub_df['pdf_links_marcia'] = sub_df['pdf_links_marcia'].str.split(';')
# sub_df = sub_df.explode('pdf_links_marcia')

# sub_df['pdf_links_marcia'] = sub_df['pdf_links_marcia'].str.strip()

# sub_df = sub_df[sub_df['pdf_links_marcia'] != '']

# print(f"Sub_df explotado: {len(sub_df):,} filas totales (una por enlace)")

# # =============================================
# # PASO 3: Contar valores únicos de id (después del explode)
# # =============================================
# unique_ids = sub_df['id'].nunique()
# print(f"Valores únicos de id: {unique_ids:,}")

# # =============================================
# # PASO 4: Guardar un CSV por cada valor de confidence_score
# #    (tratamos confidence_score como string/object, sin convertir)
# #    Nombre del archivo: 7.df_04.csv, 7.df_10.csv, 7.df_nan.csv, etc.
# # =============================================

# for score, group in sub_df.groupby('confidence_score', dropna=False):
#     if pd.isna(score):
#         score_clean = 'nan'
#         filename = '7.df_nan.csv'
#     else:
#         score_clean = str(score).replace('.', '')
#         filename = f'7.df_{score_clean}.csv'
    
#     #group.to_csv(filename, index=False)
    
#     print(f" Guardado → {filename:>20}  |  {len(group):,} filas  |  confidence_score = {score}")

# print("\n" + "="*70)
# print("PROCESO COMPLETO")
# print(f"Total de CSVs generados: {sub_df['confidence_score'].nunique(dropna=False)}")
# print("Puedes abrir directamente los archivos:")
# print("   7.df_04.csv")
# print("   7.df_10.csv")
# print("   7.df_nan.csv")
# print("   etc.")
# print("="*70)



In [ ]:
# df_04.to_csv('7.df_04.csv', index=False)
# df_042.to_csv('7.df_042.csv', index=False)
# df_06.to_csv('7.df_06.csv', index=False)
# df_065.to_csv('7.df_065.csv', index=False)
# df_07.to_csv('7.df_07.csv', index=False)
# df_nan.to_csv('7.df_nan.csv', index=False)
# df_09.to_csv('7.df_09.csv', index=False)
# df_1.to_csv('7.df_1.csv', index=False)



## Aqui vive el pdf downloader, que se alimenta del csv con los geolocalizados y va a generar un nuevo csv con los links a los pdfs y el confidence score de cada uno.

In [13]:
#df = pd.read_csv("7.df_04.csv")

In [ ]:
# import time
# import random
# from pathlib import Path

# import pandas as pd
# import requests
# import urllib3

# urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# # ============================== CONFIGURACIÓN ==============================
# CSV_FILE      = Path("7.df_04.csv")
# OUTPUT_FOLDER = Path("7.pdfs_04")
# FAILED_FILE   = Path("7.failed_downloads_04.txt")

# BATCH_SIZE        = 50
# SLEEP_PER_PDF     = (0.5, 1.5)
# SLEEP_AFTER_BATCH = 60
# MAX_RETRIES       = 3
# BACKOFF_FACTOR    = 2
# TIMEOUT           = 30


# # ======================== SISTEMA DE PROGRESO ========================
# def cargar_descargados() -> set:
#     """
#     Devuelve un set con los file stems que ya existen en disco.
#     Incluye tanto 'RM123' como 'RM123_1', 'RM123_2', etc.
#     """
#     return {f.stem for f in OUTPUT_FOLDER.glob("*.pdf")}


# def cargar_fallidos() -> set:
#     if FAILED_FILE.exists():
#         return {line.strip() for line in FAILED_FILE.read_text(encoding="utf-8").splitlines() if line.strip()}
#     return set()


# def registrar_fallido(file_stem: str):
#     with FAILED_FILE.open("a", encoding="utf-8") as f:
#         f.write(file_stem + "\n")


# # ======================== CONSTRUCCIÓN DE PENDING ========================
# def construir_pending(df: pd.DataFrame, descargados: set, fallidos: set) -> list:
#     """
#     Recorre el CSV en orden y asigna un file_stem a cada fila:
#       - Primera ocurrencia de un ID  →  ID
#       - Segunda ocurrencia           →  ID_1
#       - Tercera ocurrencia           →  ID_2  ... etc.

#     Solo incluye en pending los que no están ya en disco ni en fallidos.
#     """
#     conteo = {}   # id -> cuántas veces lo hemos visto hasta ahora
#     pending = []

#     for _, row in df.iterrows():
#         id_  = str(row["id"])
#         url  = str(row["pdf_links_marcia"])

#         n = conteo.get(id_, 0)
#         conteo[id_] = n + 1

#         # Nombre del archivo: primero sin sufijo, luego _1, _2, ...
#         file_stem = id_ if n == 0 else f"{id_}_{n}"

#         if file_stem not in descargados and file_stem not in fallidos:
#             pending.append((file_stem, url))

#     return pending


# # ======================== DESCARGA ========================
# def descargar_pdf(file_stem: str, url: str, session: requests.Session) -> bool:
#     output_path = OUTPUT_FOLDER / f"{file_stem}.pdf"

#     for intento in range(1, MAX_RETRIES + 1):
#         try:
#             response = session.get(url, timeout=TIMEOUT, verify=False, stream=True)
#             response.raise_for_status()

#             content_type = response.headers.get("Content-Type", "")
#             if "pdf" not in content_type.lower() and "octet-stream" not in content_type.lower():
#                 raise ValueError(f"Content-Type inesperado: {content_type}")

#             with output_path.open("wb") as f:
#                 for chunk in response.iter_content(chunk_size=8192):
#                     if chunk:
#                         f.write(chunk)

#             if output_path.stat().st_size == 0:
#                 output_path.unlink()
#                 raise ValueError("Archivo descargado está vacío")

#             return True

#         except Exception as e:
#             if output_path.exists():
#                 output_path.unlink()

#             if intento == MAX_RETRIES:
#                 print(f"  ✗ Falló definitivamente: {file_stem} → {e}")
#                 return False
#             else:
#                 wait = BACKOFF_FACTOR ** (intento - 1)
#                 print(f"  ↺ Intento {intento} fallido para {file_stem}, reintentando en {wait}s...")
#                 time.sleep(wait)

#     return False


# # ======================== EJECUCIÓN ========================
# def ejecutar_descarga():
#     OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

#     df = pd.read_csv(CSV_FILE)

#     if "id" not in df.columns or "pdf_links_marcia" not in df.columns:
#         raise ValueError("El CSV debe tener columnas 'id' y 'pdf_links_marcia'")

#     df = df.dropna(subset=["pdf_links_marcia"])

#     descargados = cargar_descargados()
#     fallidos    = cargar_fallidos()
#     pending     = construir_pending(df, descargados, fallidos)

#     # Info de cuántos IDs tienen múltiples PDFs
#     conteo_ids  = df["id"].astype(str).value_counts()
#     multiples   = (conteo_ids > 1).sum()

#     print(f"Total filas en CSV          : {len(df):,}")
#     print(f"IDs únicos                  : {conteo_ids.shape[0]:,}")
#     print(f"IDs con múltiples PDFs      : {multiples:,}")
#     print(f"Ya descargados              : {len(descargados):,}")
#     print(f"Fallidos definitivos        : {len(fallidos):,}")
#     print(f"Pendientes por descargar    : {len(pending):,}\n")

#     if not pending:
#         print("Todos los PDFs ya fueron descargados.")
#         return

#     start_time      = time.time()
#     nuevos          = 0
#     nuevos_fallidos = 0

#     with requests.Session() as session:
#         for i in range(0, len(pending), BATCH_SIZE):
#             batch       = pending[i:i + BATCH_SIZE]
#             batch_num   = i // BATCH_SIZE + 1
#             total_batch = (len(pending) + BATCH_SIZE - 1) // BATCH_SIZE
#             print(f"\nBatch {batch_num}/{total_batch}  ({len(batch)} PDFs)")

#             for file_stem, url in batch:
#                 ok = descargar_pdf(file_stem, url, session)

#                 if ok:
#                     nuevos += 1
#                 else:
#                     registrar_fallido(file_stem)
#                     nuevos_fallidos += 1

#                 time.sleep(random.uniform(*SLEEP_PER_PDF))

#             elapsed      = time.time() - start_time
#             velocidad    = nuevos / elapsed * 60 if elapsed > 0 else 0
#             restantes    = len(pending) - nuevos - nuevos_fallidos
#             estimado_min = (restantes / (nuevos / elapsed)) / 60 if nuevos > 0 else 0

#             print(f"  ✓ Descargados esta sesión : {nuevos:,}  |  ✗ Fallidos: {nuevos_fallidos:,}")
#             print(f"  Velocidad: {velocidad:.1f} PDFs/min  |  Estimado restante: {estimado_min:.0f} min")

#             if i + BATCH_SIZE < len(pending):
#                 print(f"  Pausa de {SLEEP_AFTER_BATCH}s...")
#                 time.sleep(SLEEP_AFTER_BATCH)

#     print(f"\n✓ Descarga completada.")
#     print(f"  Descargados esta sesión : {nuevos:,}")
#     print(f"  Fallidos definitivos    : {nuevos_fallidos:,}")
#     print(f"  PDFs totales en disco   : {len(cargar_descargados()):,}")


# if __name__ == "__main__":
#     try:
#         ejecutar_descarga()
#     except KeyboardInterrupt:
#         print("\n\nInterrumpido. Al reiniciar continuará desde donde se quedó.")

Total filas en CSV          : 9,098
IDs únicos                  : 8,456
IDs con múltiples PDFs      : 505
Ya descargados              : 379
Fallidos definitivos        : 0
Pendientes por descargar    : 8,719


Batch 1/175  (50 PDFs)


Interrumpido. Al reiniciar continuará desde donde se quedó.


In [7]:
# import pandas as pd
# import glob

# # Lista de todos tus archivos (puedes copiarla directamente)
# archivos = [
#     "7.df_042.csv",
#     "7.df_06.csv",
#     "7.df_065.csv",
#     "7.df_07.csv"
# ]

# # Opción 1: Cargar y concatenar usando la lista (recomendada)
# dfs = []
# for archivo in archivos:
#     df_temp = pd.read_csv(archivo)
#     dfs.append(df_temp)
#     print(f"✓ Cargado: {archivo}  →  {len(df_temp)} filas")

# # Concatenar todos los DataFrames
# df_pdf_resto = pd.concat(dfs, ignore_index=True)

# print("\nConcatenación terminada!")
# print(f"Total de filas: {len(df_pdf_resto)}")
# print(f"Total de columnas: {len(df_pdf_resto.columns)}")

# # Guardar el archivo combinado
# df_pdf_resto.to_csv("7_df_combinado.csv", index=False)